# 5.2 — Batch integration:

Per-config scIB benchmark on the 36 (dataset × encoder × view) configs, evaluating both the raw PCA embedding (`X_pca`) and the Harmony-integrated embedding (`X_pca_harmony`).

**Composite metrics:**
- **Batch correction** — mean of iLISI, kBET, BRAS, graph connectivity. Higher ⇒ batches well mixed.
- **Bio conservation** — mean of cLISI, NMI, ARI, silhouette label, isolated labels. Higher ⇒ perturbation labels still separable.
- **PCR comparison** — fraction of batch-attributable PCA variance removed (normalised to pre-integration reference, so `pre = 0` by construction).
- **Total** — weighted blend of the three (scIB convention: 0.4 × batch + 0.6 × bio).

Headline framing: Harmony pays a small bio-conservation cost in exchange for a large batch-correction gain. The cost is dataset-dependent; quantify it per-dataset and per-encoder.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- Global config (shared with 001_background_explotability) ---
TEXTWIDTH_PT = 397.48499  # from \the\textwidth
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27
FULLWIDTH_IN = TEXTWIDTH_IN
HALFWIDTH_IN = TEXTWIDTH_IN / 2

def neurips_style():
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "mathtext.fontset": "dejavusans",
        "font.size": 7,
        "axes.titlesize": 7,
        "axes.labelsize": 7,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.linewidth": 0.6,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "lines.linewidth": 1.0,
        "patch.linewidth": 0.6,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.01,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

neurips_style()

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

DATA_DIR = Path('../02_batch_integration')
FIG_DIR = Path('.')
FIG_DIR.mkdir(exist_ok=True)

DATASETS = ['RxRx1', 'Rxrx3C', 'JUMP']
ENCODERS = ['SubCell', 'DINO', 'OpenPhenom']
VIEWS = ['C', 'CD', 'S', 'SD']

ENCODER_DISPLAY = {
    'SubCell':    'SubCell',
    'DINO':       'DINO\nViT-S',
    'OpenPhenom': 'Open\nPhenom',
}

DATASET_LABEL = {
    'RxRx1':  'RxRx1\n(siRNA, HEPG2)',
    'Rxrx3C': 'RxRx3-core\n(CRISPR-KO, HUVEC)',
    'JUMP':   'JUMP-CP\n(compounds, U2OS)',
}

# Source CSVs are named by dataset key as it appears in evals/02_batch_integration/.
DATASET_CSV = {
    'RxRx1':  DATA_DIR / 'Rxrx1_scib_full.csv',
    'Rxrx3C': DATA_DIR / 'Rxrx3C_scib_full.csv',
    'JUMP':   DATA_DIR / 'JUMP_scib_full.csv',
}

# Composites used in the cross-dataset Δ summary table.
METRIC_COLS = ['Total', 'Batch correction', 'Bio conservation', 'PCR comparison']

# Atomic scIB metrics, grouped by which composite they contribute to.
BATCH_ATOMIC_COLS = ['iLISI', 'KBET', 'BRAS', 'Graph connectivity']
BIO_ATOMIC_COLS = ['cLISI', 'KMeans NMI', 'KMeans ARI', 'Silhouette label', 'Isolated labels']

# Full list, ordered like scIB's plot_table: atomics first, composites at end.
ALL_METRIC_COLS = (
    BATCH_ATOMIC_COLS
    + BIO_ATOMIC_COLS
    + ['PCR comparison', 'Batch correction', 'Bio conservation', 'Total']
)

In [ ]:
frames = []
for ds, path in DATASET_CSV.items():
    df = pd.read_csv(path)
    df['dataset'] = ds
    # config is e.g. 'DINO C' → split into encoder + view
    df[['encoder', 'view']] = df['config'].str.split(' ', n=1, expand=True)
    df['embedding'] = df['embedding'].map({'X_pca': 'pre_harmony', 'X_pca_harmony': 'post_harmony'})
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)
raw = raw[raw['view'].isin(VIEWS) & raw['encoder'].isin(ENCODERS)].copy()
raw['dataset'] = pd.Categorical(raw['dataset'], categories=DATASETS, ordered=True)
raw['encoder'] = pd.Categorical(raw['encoder'], categories=ENCODERS, ordered=True)
raw['view'] = pd.Categorical(raw['view'], categories=VIEWS, ordered=True)
raw[['dataset', 'encoder', 'view', 'embedding'] + METRIC_COLS].head()

In [ ]:
# Long → wide: pre/post side by side per metric, per (dataset, encoder, view).
# Pivot includes both composites and atomic metrics so the appendix tables
# can reach the same `wide` object.
wide = raw.pivot_table(
    index=['dataset', 'encoder', 'view'],
    columns='embedding',
    values=ALL_METRIC_COLS,
    observed=True,
)
# Add per-metric Harmony delta
for m in ALL_METRIC_COLS:
    wide[(m, 'delta')] = wide[(m, 'post_harmony')] - wide[(m, 'pre_harmony')]
wide = wide.sort_index(axis=1)
wide[METRIC_COLS].round(3).head(12)

In [ ]:
# Mean Harmony delta across 12 configs per dataset
delta_per_metric = pd.DataFrame({
    m: wide[(m, 'delta')].groupby('dataset', observed=True).mean()
    for m in METRIC_COLS
}).round(3)
print('Mean Harmony Δ across 12 configs per dataset:')
print(delta_per_metric)

In [ ]:
# Baseline figure stub: 3-panel (one per dataset), grouped bars
# encoder x view, post-Harmony Total. Iterate from here.

VIEW_COLOR = {
    'C':  '#2271b2',
    'CD': '#7fceaf',
    'S':  '#e84b4b',
    'SD': '#ff913a',
}

bar_w = 1.0
group_inner_w = len(VIEWS) * bar_w
group_gap = 1.5
group_step = group_inner_w + group_gap

encoder_centers = np.array([
    ei * group_step + (group_inner_w - bar_w) / 2
    for ei in range(len(ENCODERS))
])
bar_x = {
    (enc, view): ei * group_step + vi * bar_w
    for ei, enc in enumerate(ENCODERS)
    for vi, view in enumerate(VIEWS)
}


def label_panel(ax, label, x=-0.15, y=1.2):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top', ha='left')


def render_metric_figure(metric: str, stage: str, out_stem: str, ymax: float | None = None):
    """3-panel grouped-bar figure for a single scIB metric and Harmony stage."""
    df = raw[raw['embedding'] == stage]
    if ymax is None:
        ymax = df[metric].max() * 1.05

    fig, axes = plt.subplots(1, 3, figsize=(FULLWIDTH_IN, 2))
    for ax, ds in zip(axes, DATASETS):
        sub = df[df['dataset'] == ds].set_index(['encoder', 'view'])
        for enc in ENCODERS:
            for view in VIEWS:
                ax.bar(
                    bar_x[(enc, view)],
                    sub.loc[(enc, view), metric],
                    width=bar_w,
                    color=VIEW_COLOR[view],
                    edgecolor='black',
                    linewidth=0.4,
                )
        all_x = [bar_x[(enc, v)] for enc in ENCODERS for v in VIEWS]
        ax.set_xticks(all_x)
        ax.set_xticklabels([''] * len(all_x))
        for centre, enc in zip(encoder_centers, ENCODERS):
            ax.text(centre, -0.10, ENCODER_DISPLAY[enc],
                    transform=ax.get_xaxis_transform(),
                    ha='center', va='top')
        ax.set_title(DATASET_LABEL[ds])
        ax.axhline(0, color='gray', linewidth=0.5)
        ax.set_xlim(-bar_w, encoder_centers[-1] + group_inner_w / 2 + bar_w / 2)
        ax.set_ylim(0, ymax)

    for ax, lbl in zip(axes, 'ABC'):
        label_panel(ax, lbl)

    legend_handles = [Patch(color=VIEW_COLOR[v], label=v) for v in VIEWS]
    axes[2].legend(
        handles=legend_handles,
        loc='upper right', frameon=False, fontsize=7,
        handlelength=1.0, handleheight=1.0,
        borderpad=0.1, labelspacing=0.1,
    )

    stage_label = 'Post-Harmony' if stage == 'post_harmony' else 'Pre-Harmony'
    axes[0].set_ylabel(f'{metric} ({stage_label})')
    fig.tight_layout()

    out_pdf = FIG_DIR / f'{out_stem}.pdf'
    fig.savefig(out_pdf)
    fig.savefig(out_pdf.with_suffix('.png'), dpi=300)
    plt.show()
    print(f'Saved: {out_pdf}')


# Baseline call: post-Harmony Total. Iterate from here.
render_metric_figure('Total', 'post_harmony', 'fig3_scib_total_post_harmony')

In [ ]:
DATASET_TEX = {'RxRx1': 'RxRx1', 'Rxrx3C': 'RxRx3-core', 'JUMP': 'JUMP-CP'}

# Main-text table: cross-dataset Harmony delta summary (markdown)
print('## Cross-dataset Harmony delta summary')
print()
print('Mean Harmony delta across the 12 (encoder × view) configurations per dataset.')
print()
print('| Dataset | Δ Total | Δ Batch correction | Δ Bio conservation | Δ PCR comparison |')
print('|---|---|---|---|---|')
for ds in DATASETS:
    row = delta_per_metric.loc[ds]
    print(f"| {DATASET_TEX[ds]} | "
          f"{row['Total']:+.3f} | "
          f"{row['Batch correction']:+.3f} | "
          f"{row['Bio conservation']:+.3f} | "
          f"{row['PCR comparison']:+.3f} |")

In [ ]:
def render_full_table_md(
    stage: str,
    title: str,
    bold_best_total: bool = False,
    metric_cols: list[str] = METRIC_COLS,
) -> None:
    """Print markdown table of all 12 configs per dataset.

    stage:        'pre_harmony' | 'post_harmony'
    metric_cols:  which scIB metrics to include as columns
    """
    print(f'## {title}')
    print()
    header_metrics = ' | '.join(metric_cols)
    print(f'| Dataset | Encoder | View | {header_metrics} |')
    print('|' + '|'.join(['---'] * (3 + len(metric_cols))) + '|')

    total_idx = metric_cols.index('Total') if 'Total' in metric_cols else None

    for ds in DATASETS:
        ds_block = wide.loc[ds]
        best_idx = ds_block[('Total', stage)].idxmax() if bold_best_total else None

        for (enc, view), row in ds_block.iterrows():
            cells = [f'{row[(m, stage)]:.3f}' for m in metric_cols]
            if best_idx is not None and (enc, view) == best_idx and total_idx is not None:
                cells[total_idx] = f'**{cells[total_idx]}**'
            print(f'| {DATASET_TEX[ds]} | {enc} | {view} | ' + ' | '.join(cells) + ' |')
    print()


# Appendix: post-Harmony, all metrics, bold best Total per dataset
render_full_table_md(
    'post_harmony',
    'Full scIB scores, post-Harmony (best Total per dataset bolded)',
    bold_best_total=True,
    metric_cols=ALL_METRIC_COLS,
)

In [ ]:
# Appendix: pre-Harmony, all metrics
render_full_table_md(
    'pre_harmony',
    'Full scIB scores, pre-Harmony (best Total per dataset bolded)',
    bold_best_total=True,
    metric_cols=ALL_METRIC_COLS,
)

In [ ]:
# Appendix figure: Batch correction vs bio conservation scatter
# One panel per dataset; view = colour (matches bar-chart convention),
# encoder = marker; open = pre, filled = post.
# Thin grey arrow from pre → post for each (encoder, view) pair.

# Reuse VIEW_COLOR from the baseline figure cell for visual consistency.
MARKER_BY_ENCODER = {
    'SubCell':    'o',
    'DINO':       's',
    'OpenPhenom': '^',
}

# Keep figsize at FULLWIDTH_IN so the saved PDF imports at 1.0× in LaTeX —
# 7pt fonts then render at 7pt on the page. Reserve the right band of the
# figure for the legends via subplots_adjust + figure-anchored legends, AND
# turn off bbox_inches='tight' for this savefig so the right band isn't
# re-cropped away (rcParam default would otherwise trim to artist extents).
fig, axes = plt.subplots(1, 3, figsize=(FULLWIDTH_IN, 1.5), sharex=True, sharey=True)
for ax, ds in zip(axes, DATASETS):
    ds_block = wide.loc[ds]
    for (enc, view), row in ds_block.iterrows():
        pre_bc = row[('Batch correction', 'pre_harmony')]
        pre_bio = row[('Bio conservation', 'pre_harmony')]
        post_bc = row[('Batch correction', 'post_harmony')]
        post_bio = row[('Bio conservation', 'post_harmony')]
        color = VIEW_COLOR[view]
        marker = MARKER_BY_ENCODER[enc]
        ax.annotate(
            '', xy=(post_bc, post_bio), xytext=(pre_bc, pre_bio),
            arrowprops=dict(arrowstyle='->', color='gray', lw=0.4, alpha=0.5),
        )
        ax.scatter(pre_bc, pre_bio, marker=marker, facecolor='none',
                   edgecolor=color, s=22, lw=0.7)
        ax.scatter(post_bc, post_bio, marker=marker, facecolor=color,
                   edgecolor='black', s=22, lw=0.4)

    ax.set_xlabel('Batch correction')
    ax.set_title(DATASET_LABEL[ds])
    ax.grid(alpha=0.25, lw=0.3)

axes[0].set_ylabel('Bio conservation')

view_handles = [
    plt.Line2D([], [], marker='o', color=c, lw=0, label=v, markersize=4.5,
               markeredgecolor='black', markeredgewidth=0.4)
    for v, c in VIEW_COLOR.items()
]
encoder_handles = [
    plt.Line2D([], [], marker=m, color='black', lw=0, label=enc, markersize=4.5,
               markerfacecolor='none', markeredgewidth=0.7)
    for enc, m in MARKER_BY_ENCODER.items()
]
state_handles = [
    plt.Line2D([], [], marker='o', color='gray', lw=0, label='pre', markersize=4.5,
               markerfacecolor='none', markeredgewidth=0.7),
    plt.Line2D([], [], marker='o', color='gray', lw=0, label='post', markersize=4.5,
               markeredgecolor='black', markeredgewidth=0.4),
]


def _left_align_title(legend):
    """matplotlib centres legend titles by default; force left-alignment."""
    legend._legend_box.align = 'left'
    return legend


# Physically constrain axes to the left 72%; legends sit in the right 28%
# (≈1.5") band, in figure coordinates, so they're independent of axes[2].
fig.subplots_adjust(left=0.07, right=0.72, bottom=0.18, top=0.88, wspace=0.18)

legend_kwargs = dict(
    loc='upper left',
    frameon=False,
    fontsize=6,
    title_fontsize=6,
    handletextpad=0.4,
    borderpad=0.2,
    labelspacing=0.25,
    bbox_transform=fig.transFigure,
)
_left_align_title(fig.legend(
    handles=view_handles, title='View',
    bbox_to_anchor=(0.74, 1.05), **legend_kwargs,
))
_left_align_title(fig.legend(
    handles=encoder_handles, title='Encoder',
    bbox_to_anchor=(0.74, 0.63), **legend_kwargs,
))
_left_align_title(fig.legend(
    handles=state_handles, title='Harmony',
    bbox_to_anchor=(0.74, 0.27), **legend_kwargs,
))

for ax, lbl in zip(axes, 'ABC'):
    label_panel(ax, lbl)

# Override the rcParam bbox='tight' for this save so the reserved-right band
# isn't cropped away. Pad slightly for breathing room.
out_pdf = FIG_DIR / 'fig_batch_integration_scatter.pdf'
fig.savefig(out_pdf, bbox_inches=None, pad_inches=0.05)
fig.savefig(out_pdf.with_suffix('.png'), dpi=300, bbox_inches=None, pad_inches=0.05)
plt.show()
print(f'Saved: {out_pdf}')